In [10]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

# =================== 配置区域 ===================
file_path = 'senna_cohort.csv'          # 修改为你的数据路径
exposed_col = 'senna_exposed'           # 暴露变量列名
outcome_col = 'hypokalemia_event'       # 结局变量列名
var = 'drug_count_no_laxatives'         # 非泻药类合并用药数列名
# =================================================

# 读取数据
df = pd.read_csv(file_path)

def format_pvalue(p):
    """P值格式化为 x10的几次方，若过小则自动增加精度，避免显示0.00×10⁰"""
    if p <= 0:
        return "<0.001"                 # 极小数或下溢为0时统一显示
    if p < 0.001:
        # 先尝试保留2位小数，如果尾数变成0.00，则保留更多位
        mantissa, exp = f"{p:.2e}".split("e")
        if mantissa in ("0.00", "0.01", "0.99"):  # 极端情况再提高精度
            mantissa, exp = f"{p:.4e}".split("e")
        exp = int(exp)
        # 转换为上标字符，如 10⁻²¹
        superscripts = {
            "0": "⁰", "1": "¹", "2": "²", "3": "³", "4": "⁴",
            "5": "⁵", "6": "⁶", "7": "⁷", "8": "⁸", "9": "⁹",
            "-": "⁻"
        }
        exp_str = "".join(superscripts[ch] for ch in str(exp))
        return f"{mantissa}×10{exp_str}"
    else:
        return f"{p:.4f}"

# ---------- 1. 基线表统计 ----------
group_exp = df.loc[df[exposed_col] == 1, var].dropna()
group_unexp = df.loc[df[exposed_col] == 0, var].dropna()

def describe(s):
    mean = s.mean()
    sd = s.std(ddof=1)
    median = s.median()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    # 样本量<5000时做Shapiro正态性检验，否则近似非正态
    if len(s) < 5000:
        _, p_shapiro = stats.shapiro(s)
    else:
        p_shapiro = 0.001
    return mean, sd, median, q1, q3, p_shapiro

mean_exp, sd_exp, med_exp, q1_exp, q3_exp, pnorm_exp = describe(group_exp)
mean_unexp, sd_unexp, med_unexp, q1_unexp, q3_unexp, pnorm_unexp = describe(group_unexp)

if pnorm_exp < 0.05 or pnorm_unexp < 0.05:
    p_base = stats.mannwhitneyu(group_exp, group_unexp, alternative='two-sided').pvalue
    test_used = "Mann-Whitney U"
    summary_exp = f"{med_exp} ({q1_exp}-{q3_exp})"
    summary_unexp = f"{med_unexp} ({q1_unexp}-{q3_unexp})"
else:
    p_base = stats.ttest_ind(group_exp, group_unexp, equal_var=False).pvalue
    test_used = "Welch t-test"
    summary_exp = f"{mean_exp:.2f} ± {sd_exp:.2f}"
    summary_unexp = f"{mean_unexp:.2f} ± {sd_unexp:.2f}"

smd = (mean_exp - mean_unexp) / np.sqrt((sd_exp**2 + sd_unexp**2) / 2)

print("=" * 60)
print("基线表结果：非泻药类合并用药数")
print("=" * 60)
print(f"暴露组 (senna=1): {summary_exp}")
print(f"未暴露组 (senna=0): {summary_unexp}")
print(f"检验方法: {test_used}")
print(f"P值: {format_pvalue(p_base)}")
print(f"SMD: {smd:.4f}")
print(f"P<0.05: {'P<0.05' if p_base < 0.05 else 'P>0.05'}")

# ---------- 2. 单变量逻辑回归 ----------
data_model = df[[outcome_col, var]].dropna()
X = data_model[var]
y = data_model[outcome_col]

X_const = sm.add_constant(X)
model = sm.Logit(y, X_const).fit(disp=False)

coef = model.params[var]
se = model.bse[var]
OR = np.exp(coef)
ci_low = np.exp(coef - 1.96 * se)
ci_high = np.exp(coef + 1.96 * se)
p_reg = model.pvalues[var]

print("\n" + "=" * 60)
print("单变量逻辑回归结果")
print("=" * 60)
print(f"结局变量: {outcome_col}")
print(f"变量: {var}")
print(f"OR: {OR:.4f}")
print(f"95% CI: {ci_low:.4f} - {ci_high:.4f}")
print(f"P值: {format_pvalue(p_reg)}")
print(f"P<0.05: {'P<0.05' if p_reg < 0.05 else 'P>0.05'}")

基线表结果：非泻药类合并用药数
暴露组 (senna=1): 11.0 (8.0-15.0)
未暴露组 (senna=0): 11.0 (8.0-14.0)
检验方法: Mann-Whitney U
P值: 5.72×10⁻⁷¹
SMD: 0.2030
P<0.05: P<0.05

单变量逻辑回归结果
结局变量: hypokalemia_event
变量: drug_count_no_laxatives
OR: 1.1135
95% CI: 1.1083 - 1.1187
P值: <0.001
P<0.05: P<0.05
